# 🧠 K8sGPT Dashboard

In [3]:
import subprocess
import json
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# List all namespaces
def get_namespaces():
    try:
        result = subprocess.run(
            ["kubectl", "get", "ns", "-o", "jsonpath={.items[*].metadata.name}"],
            capture_output=True,
            text=True,
            check=True
        )
        return result.stdout.strip().split()
    except subprocess.CalledProcessError:
        return []

# Run k8sgpt
def run_k8sgpt(namespace=None, explain=True):
    cmd = ["k8sgpt", "analyze", "--output", "json"]
    if explain:
        cmd.extend(["--explain", "--backend", "ollama"])
    if namespace and namespace != "All":
        cmd.extend(["-n", namespace])  # Fixed here

    try:
        print("Running:", ' '.join(cmd))
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        return json.loads(result.stdout)
    except subprocess.CalledProcessError as e:
        print("❌ Error running k8sgpt:\n", e.stderr)
        return None

# Display data
def display_results(data):
    if not data or "results" not in data or not data["results"]:
        display(Markdown("### ✅ No issues found!"))
        return

    df = pd.json_normalize(data["results"])
    for col in ["namespace", "reason", "explanation"]:
        if col not in df.columns:
            df[col] = "N/A"

    pd.set_option('display.max_colwidth', None)
    display(Markdown("## 🔍 K8sGPT Analysis Results"))
    display(df[["name", "kind", "namespace", "reason", "explanation"]])

    kind_count = df["kind"].value_counts().reset_index()
    kind_count.columns = ["Resource Kind", "Count"]
    fig = px.bar(kind_count, x="Resource Kind", y="Count", title="Issues by Resource Type", color="Count")
    fig.show()

    def export_csv(_):
        df.to_csv("k8sgpt_analysis.csv", index=False)
        print("✅ Saved to k8sgpt_analysis.csv")

    export_button = widgets.Button(description="📁 Export to CSV")
    export_button.on_click(export_csv)
    display(export_button)

# Widgets
namespaces = ["All"] + get_namespaces()
namespace_dropdown = widgets.Dropdown(options=namespaces, description="Namespace:")
run_button = widgets.Button(description="🔍 Run Analysis")
output_area = widgets.Output()

def on_run_clicked(b):
    with output_area:
        clear_output()
        ns = namespace_dropdown.value
        data = run_k8sgpt(namespace=ns)
        display_results(data)

run_button.on_click(on_run_clicked)

display(widgets.HBox([namespace_dropdown, run_button]))
display(output_area)


Output()